# Nearest-Neighbor Distance (NND) Privacy Evaluation

For each real training record, find the closest synthetic record (L2 distance in min-max-normalised feature space) and report summary statistics.

**Interpretation:** A large mean/median NND means the generator is NOT copying real records verbatim → privacy-preserving. A very small NND (especially near zero) signals near-duplicate records → privacy risk.

## 1. Imports & Paths

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

BASE = Path(".").resolve().parent          # thesis/
DATA = BASE / "data"
OUT  = Path(".").resolve() / "results"
OUT.mkdir(exist_ok=True)

REAL_TRAIN = DATA / "diabetic_data_preprocessed_train.csv"
MIN_MAX    = DATA / "diabetic_data_preprocessed_train.min_max"

SYNTHETIC = {
    "HealthGAN": [
        DATA / "healthgan" / f"samples_99999_5_64_synthetic_{i}.csv"
        for i in range(10)
    ],
    "PATEGAN_train": [DATA / "pategan" / "diabetic_data_pategan_train_synthetic.csv"],
    "PATEGAN_test":  [DATA / "pategan" / "diabetic_data_pategan_test_synthetic.csv"],
    "Pythia_train":  [DATA / "pythia"  / "diabetic_data_pythia_train_synthetic.csv"],
    "Pythia_test":   [DATA / "pythia"  / "diabetic_data_pythia_test_synthetic.csv"],
}

DROP_COLS = ["encounter_id"]   # identifier — not a feature
print("Paths ready.")

Paths ready.


## 2. Helper Functions

In [16]:
def load_min_max(path: Path) -> dict:
    with open(path) as f:
        return json.load(f)


def get_feature_cols(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in DROP_COLS]


def normalise(df: pd.DataFrame, min_max: dict, feature_cols: list) -> np.ndarray:
    """Min-max normalise to [0,1]. Columns absent from min_max (binary flags) are left as-is."""
    arr = df[feature_cols].copy().astype(float)
    for col in feature_cols:
        if col in min_max:
            lo, hi = min_max[col][0], min_max[col][1]
            rng = hi - lo
            if rng > 0:
                arr[col] = (arr[col] - lo) / rng
    return arr.values


def nearest_neighbor_distances(real: np.ndarray, synthetic: np.ndarray) -> np.ndarray:
    """For every real record, return the L2 distance to its nearest synthetic neighbour."""
    nn = NearestNeighbors(n_neighbors=1, algorithm="auto", metric="euclidean", n_jobs=-1)
    nn.fit(synthetic)
    distances, _ = nn.kneighbors(real)
    return distances.ravel()


def summarise(distances: np.ndarray) -> dict:
    return {
        "mean":   float(np.mean(distances)),
        "median": float(np.median(distances)),
        "std":    float(np.std(distances)),
        "min":    float(np.min(distances)),
        "max":    float(np.max(distances)),
        "p5":     float(np.percentile(distances, 5)),
        "p25":    float(np.percentile(distances, 25)),
        "p75":    float(np.percentile(distances, 75)),
        "p95":    float(np.percentile(distances, 95)),
    }

print("Helpers defined.")

Helpers defined.


## 3. Load & Normalise Real Training Data

In [3]:
real_df   = pd.read_csv(REAL_TRAIN)
min_max   = load_min_max(MIN_MAX)
feat_cols = get_feature_cols(real_df)

real_norm = normalise(real_df, min_max, feat_cols)
print(f"Real records : {len(real_norm):,}")
print(f"Features     : {real_norm.shape[1]}")
real_df[feat_cols].head(3)

Real records : 57,214
Features     : 44


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,0,1.5,2,0,0,2.0,33.0,0.0,8.0,0.0,...,0,1,0,0,0,0,0,1,1,0
1,1,1.5,2,0,0,3.0,1.0,0.0,17.0,2.0,...,0,1,0,0,0,0,0,0,1,1
2,0,1.5,0,0,14,2.0,52.0,0.0,20.0,2.0,...,0,0,0,0,0,0,0,1,1,1


## 4. NND — HealthGAN

In [4]:
model_name = "HealthGAN"
frames = []
for p in SYNTHETIC[model_name]:
    df = pd.read_csv(p)
    df = df[[c for c in feat_cols if c in df.columns]]
    frames.append(df)

synth_healthgan = pd.concat(frames, ignore_index=True)
# HealthGAN outputs are already min-max normalised
synth_healthgan_norm = synth_healthgan.values.astype(float)

print(f"Synthetic records: {len(synth_healthgan_norm):,}")
synth_healthgan.head(3)

Synthetic records: 100,000


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,0.278101,5.052892e-01,0.447755,0.260774,0.213650,0.186321,0.258836,6.821793e-01,0.170194,0.001420,...,0.502829,0.334158,0.465808,0.650819,0.571517,0.610799,0.379605,0.671343,0.494509,0.722814
1,0.254267,2.035144e-24,0.147237,0.314104,0.797851,0.050024,0.325038,9.504492e-07,0.101585,0.000010,...,0.463603,0.701477,0.215517,0.387929,0.478849,0.672696,0.771914,0.215585,0.864669,0.312024
2,0.728084,2.744812e-03,0.690000,0.564030,0.426586,0.004158,0.265756,4.385420e-01,0.040712,0.000004,...,0.594593,0.873349,0.558937,0.493098,0.550321,0.451333,0.420589,0.359438,0.902075,0.316258


In [5]:
nnd_healthgan = nearest_neighbor_distances(real_norm, synth_healthgan_norm)
stats_healthgan = summarise(nnd_healthgan)

print(f"Mean NND  : {stats_healthgan['mean']:.4f}")
print(f"Median NND: {stats_healthgan['median']:.4f}")
print(f"Std       : {stats_healthgan['std']:.4f}")
print(f"Min NND   : {stats_healthgan['min']:.6f}")
print(f"Max NND   : {stats_healthgan['max']:.4f}")
print(f"P5 / P95  : {stats_healthgan['p5']:.4f} / {stats_healthgan['p95']:.4f}")

Mean NND  : 15.5793
Median NND: 15.1767
Std       : 6.2359
Min NND   : 2.470924
Max NND   : 29.6660
P5 / P95  : 4.9026 / 25.6935


## 5. NND — PATEGAN (train split)

In [6]:
model_name = "PATEGAN_train"
synth_pategan_train = pd.read_csv(SYNTHETIC[model_name][0])
synth_pategan_train = synth_pategan_train[[c for c in feat_cols if c in synth_pategan_train.columns]]
synth_pategan_train_norm = normalise(synth_pategan_train, min_max, feat_cols)

print(f"Synthetic records: {len(synth_pategan_train_norm):,}")
synth_pategan_train.head(3)

Synthetic records: 57,214


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,0,1.5,0,9,14,8.766182,92.56976,3.604761,27.960838,24.922007,...,0,0,0,0,0,0,0,0,1,0
1,0,1.5,1,0,14,9.149098,49.24236,2.246476,72.275780,17.842426,...,0,1,0,0,0,0,0,1,1,0
2,0,1.5,0,19,14,10.869281,100.59687,3.281893,21.201233,19.690092,...,0,1,0,0,0,0,0,1,1,0


In [7]:
nnd_pategan_train = nearest_neighbor_distances(real_norm, synth_pategan_train_norm)
stats_pategan_train = summarise(nnd_pategan_train)

print(f"Mean NND  : {stats_pategan_train['mean']:.4f}")
print(f"Median NND: {stats_pategan_train['median']:.4f}")
print(f"Std       : {stats_pategan_train['std']:.4f}")
print(f"Min NND   : {stats_pategan_train['min']:.6f}")
print(f"Max NND   : {stats_pategan_train['max']:.4f}")
print(f"P5 / P95  : {stats_pategan_train['p5']:.4f} / {stats_pategan_train['p95']:.4f}")

Mean NND  : 2.1746
Median NND: 1.9776
Std       : 0.8037
Min NND   : 0.601226
Max NND   : 6.8160
P5 / P95  : 1.2510 / 3.8157


## 6. NND — PATEGAN (test split)

In [8]:
model_name = "PATEGAN_test"
synth_pategan_test = pd.read_csv(SYNTHETIC[model_name][0])
synth_pategan_test = synth_pategan_test[[c for c in feat_cols if c in synth_pategan_test.columns]]
synth_pategan_test_norm = normalise(synth_pategan_test, min_max, feat_cols)

print(f"Synthetic records: {len(synth_pategan_test_norm):,}")
synth_pategan_test.head(3)

Synthetic records: 14,304


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,0,1.5,1,0,5,5,82,3,51,6,...,0,0,0,0,0,0,0,0,1,0
1,0,1.5,1,0,14,11,59,3,43,4,...,0,1,0,0,0,0,0,0,1,0
2,1,1.5,0,0,14,9,47,4,55,6,...,0,1,0,0,0,0,0,1,1,0


In [9]:
nnd_pategan_test = nearest_neighbor_distances(real_norm, synth_pategan_test_norm)
stats_pategan_test = summarise(nnd_pategan_test)

print(f"Mean NND  : {stats_pategan_test['mean']:.4f}")
print(f"Median NND: {stats_pategan_test['median']:.4f}")
print(f"Std       : {stats_pategan_test['std']:.4f}")
print(f"Min NND   : {stats_pategan_test['min']:.6f}")
print(f"Max NND   : {stats_pategan_test['max']:.4f}")
print(f"P5 / P95  : {stats_pategan_test['p5']:.4f} / {stats_pategan_test['p95']:.4f}")

Mean NND  : 2.3183
Median NND: 2.0584
Std       : 0.9329
Min NND   : 0.412939
Max NND   : 8.0155
P5 / P95  : 1.2527 / 4.2184


## 7. NND — Pythia (train split)

In [17]:
model_name = "Pythia_train"
synth_pythia_train = pd.read_csv(SYNTHETIC[model_name][0])
synth_pythia_train = synth_pythia_train[[c for c in feat_cols if c in synth_pythia_train.columns]]
synth_pythia_train_norm = normalise(synth_pythia_train, min_max, feat_cols)

print(f"Synthetic records: {len(synth_pythia_train_norm):,}")
synth_pythia_train.head(3)

Synthetic records: 10


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,0,1.5,0,22,14,1.0,9.0,0.0,14.0,2.0,...,0,1,0,0,0,0,0,0,1,1
1,0,2.0,4,9,0,2.0,1.0,2.0,27.0,0.0,...,0,1,0,0,0,0,0,1,1,0
2,0,2.0,4,19,14,4.0,1.0,2.0,26.0,0.0,...,0,0,0,0,0,0,0,1,0,0


In [11]:
nnd_pythia_train = nearest_neighbor_distances(real_norm, synth_pythia_train_norm)
stats_pythia_train = summarise(nnd_pythia_train)

print(f"Mean NND  : {stats_pythia_train['mean']:.4f}")
print(f"Median NND: {stats_pythia_train['median']:.4f}")
print(f"Std       : {stats_pythia_train['std']:.4f}")
print(f"Min NND   : {stats_pythia_train['min']:.6f}")
print(f"Max NND   : {stats_pythia_train['max']:.4f}")
print(f"P5 / P95  : {stats_pythia_train['p5']:.4f} / {stats_pythia_train['p95']:.4f}")

Mean NND  : 10.2824
Median NND: 10.8214
Std       : 2.8860
Min NND   : 1.460386
Max NND   : 16.2746
P5 / P95  : 4.8401 / 14.3071


## 8. NND — Pythia (test split)

In [12]:
model_name = "Pythia_test"
synth_pythia_test = pd.read_csv(SYNTHETIC[model_name][0])
synth_pythia_test = synth_pythia_test[[c for c in feat_cols if c in synth_pythia_test.columns]]
synth_pythia_test_norm = normalise(synth_pythia_test, min_max, feat_cols)

print(f"Synthetic records: {len(synth_pythia_test_norm):,}")
synth_pythia_test.head(3)

Synthetic records: 10


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,0,1.0,2,18,0,6.0,38.0,2.0,6.0,1.0,...,0,0,0,0,0,0,0,0,0,0
1,1,1.5,4,19,0,5.0,61.0,0.0,30.0,1.0,...,0,0,0,0,0,0,0,0,0,1
2,1,1.0,2,21,5,11.0,36.0,6.0,30.0,0.0,...,0,0,0,0,0,0,0,0,1,0


In [13]:
nnd_pythia_test = nearest_neighbor_distances(real_norm, synth_pythia_test_norm)
stats_pythia_test = summarise(nnd_pythia_test)

print(f"Mean NND  : {stats_pythia_test['mean']:.4f}")
print(f"Median NND: {stats_pythia_test['median']:.4f}")
print(f"Std       : {stats_pythia_test['std']:.4f}")
print(f"Min NND   : {stats_pythia_test['min']:.6f}")
print(f"Max NND   : {stats_pythia_test['max']:.4f}")
print(f"P5 / P95  : {stats_pythia_test['p5']:.4f} / {stats_pythia_test['p95']:.4f}")

Mean NND  : 10.9794
Median NND: 9.6031
Std       : 4.4676
Min NND   : 2.540596
Max NND   : 19.1060
P5 / P95  : 5.1893 / 17.3019


## 9. Summary Table & Save

In [14]:
all_results = {
    "HealthGAN":     {"distances": nnd_healthgan,      "stats": stats_healthgan},
    "PATEGAN_train": {"distances": nnd_pategan_train,  "stats": stats_pategan_train},
    "PATEGAN_test":  {"distances": nnd_pategan_test,   "stats": stats_pategan_test},
    "Pythia_train":  {"distances": nnd_pythia_train,   "stats": stats_pythia_train},
    "Pythia_test":   {"distances": nnd_pythia_test,    "stats": stats_pythia_test},
}

rows = [{"model": k, **v["stats"]} for k, v in all_results.items()]
summary_df = pd.DataFrame(rows).set_index("model").round(4)

# Save CSV
summary_df.to_csv(OUT / "nnd_summary.csv")

# Save per-record distance arrays
for name, data in all_results.items():
    np.save(OUT / f"nnd_{name}.npy", data["distances"])

print(f"Saved to {OUT}")
summary_df

Saved to /Users/kazimostafashahriar/Main Drive/thesis/Medical data analysis/Code/thesis/privacy_evaluation/results


,mean,median,std,min,max,p5,p25,p75,p95
model,,,,,,,,,
HealthGAN,15.5793,15.1767,6.2359,2.4709,29.6660,4.9026,11.0405,20.6672,25.6935
PATEGAN_train,2.1746,1.9776,0.8037,0.6012,6.8160,1.2510,1.6493,2.4710,3.8157
PATEGAN_test,2.3183,2.0584,0.9329,0.4129,8.0155,1.2527,1.6767,2.7360,4.2184
Pythia_train,10.2824,10.8214,2.8860,1.4604,16.2746,4.8401,9.1372,12.2247,14.3071
Pythia_test,10.9794,9.6031,4.4676,2.5406,19.1060,5.1893,6.7384,15.6698,17.3019


In [15]:
len(synth_pythia_train)

10